# Model Versions & Experiment History Tracker

This notebook records the progression of the Aimpathy AI Sign Language Recognition models, dataset splits, evaluation metrics, and engineering decisions leading up to the stable `asl_ctc_v6.pth` release checkpoint.

## Key Milestones
* **Baseline (`SignNet` + `SignDecider`):** Frame-window classifier with rule-based smoothing. Effective for isolated numbers (`123`) and static words, but achieved **73.8% CER** and **1.1% exact-word accuracy** on continuous fingerspelled words (`data/words_test`).
* **CTC Sequence Pipeline (`SeqNet` + `StreamingDecoder`):** Introduced end-to-end temporal sequence modeling over 4-second windows (`T` frames) trained on dynamically stitched letter sequences (`data/takes`) plus real word recordings (`data/words_train`).
* **Stable Release (`asl_ctc_v6.pth`):** Verified stable during live webcam inference (`scripts/recognize_asl.py`) and benchmarked on 91 held-out real word takes (`data/words_test`, 30 unique words, signer `leyu_test`), cutting CER by more than half to **34.5%** and raising exact-word accuracy to **15.4%**.

In [ ]:
from pathlib import Path
import sys

# Resolve project root whether launched from repo root or inside notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root : {PROJECT_ROOT}")
print(f"src path     : {SRC_DIR} (exists={SRC_DIR.exists()})")

## 1. Experiment Comparison Table
Log new training runs (`v7`, fine-tuned checkpoints, hyperparameter updates) below alongside their `data/words_test` scores.

In [ ]:
import pandas as pd

experiments = [
    {
        "Checkpoint": "Baseline (SignNet + rules)",
        "Location": "Legacy (archived)",
        "Training Data": "Isolated takes (data/takes)",
        "Eval Set": "data/words_test (91 takes, 30 words)",
        "CER (%)": 73.8,
        "Exact Word (%)": 1.1,
        "Status": "Superseded",
        "Notes": "Drops middle/repeated letters on continuous fingerspelling.",
    },
    {
        "Checkpoint": "asl_ctc_v3 - v5",
        "Location": "Earlier iterations",
        "Training Data": "Stitched sequences (data/takes)",
        "Eval Set": "data/words_test (91 takes, 30 words)",
        "CER (%)": None,
        "Exact Word (%)": None,
        "Status": "Archived",
        "Notes": "Early CTC iterations before real-word mixing and stability tuning.",
    },
    {
        "Checkpoint": "asl_ctc_v6.pth",
        "Location": "trained_models/ctc/asl_ctc_v6.pth",
        "Training Data": "data/takes + data/words_train",
        "Eval Set": "data/words_test (91 takes, 30 words)",
        "CER (%)": 34.5,
        "Exact Word (%)": 15.4,
        "Status": "Production / Stable",
        "Notes": "More stable in live webcam streaming; 3/3 on LOVE & TEA, 2/3 on QUICK & WORK.",
    },
    {
        "Checkpoint": "asl_ctc_v7.pth",
        "Location": "Experimental draft",
        "Training Data": "data/takes + data/words_train",
        "Eval Set": "data/words_test (91 takes, 30 words)",
        "CER (%)": None,
        "Exact Word (%)": None,
        "Status": "Experimental",
        "Notes": "Less stable in live streaming than v6; retained v6 as main checkpoint.",
    },
]

df_history = pd.DataFrame(experiments)
df_history

## 2. Error Analysis & Next Improvements for `v7+`
Based on `eval_words.py --verbose` output for `asl_ctc_v6.pth`:
1. **Repeated/Inserted Characters:** Words like `APPLE` (`AAPXLE`), `EXAM` (`BEEXXAA`), and `SUMMER` (`SUUESSER`) emit extra vowel/consonant repeats when hand hold duration varies.
   * *Next step:* Tune blank-bias / duplicate suppression in `src/stream_decoder.py` and increase temporal speed jitter augmentation in `src/seq_data.py`.
2. **Look-Alike Handshapes:** Confusions such as `M`/`N` (`MILK` -> `ALK`, `MOON` -> `SCOS`) and `Z`/`X` (`ZEBRA` -> `XEBRA`).
   * *Next step:* Collect additional real continuous word takes (`scripts/collect_words.py`) with `--exclude-words-dir data/words_test` enforced during training.

In [ ]:
# Quick verification that the active release checkpoint exists in the expected template path
release_ckpt = PROJECT_ROOT / "trained_models" / "ctc" / "asl_ctc_v6.pth"
assert release_ckpt.exists(), f"Missing release checkpoint: {release_ckpt}"
size_kb = release_ckpt.stat().st_size / 1024
print(f"Verified release model: {release_ckpt.name} ({size_kb:.1f} KB)")